# 人手アノテーションによる SFT（Supervised Fine-Tuning）

疑似事前学習済み BERT モデルをベースに、人手アノテーション CSV で
PROFILE / PII / QPII / NONE の 4 ラベル分類を BIO tagging として学習する。

- Stage 1（疑似事前学習）: O / B-LABEL / I-LABEL の 3 クラス
- Stage 2（本ノートブック）: O / B-PROFILE / I-PROFILE / B-PII / I-PII / B-QPII / I-QPII の 7 クラス

## Google Drive マウント

疑似事前学習済みモデル・アノテーション CSV の読み込みと、SFT モデルの保存先を Google Drive に設定する。

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 依存パッケージのインストール

In [2]:
!pip install datasets pydantic transformers seqeval scikit-learn fugashi unidic-lite wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 44.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 694.9/694.9 kB 57.5 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=4502ae492846685e76baa15a9d7451cd77c85dee50c83adbdd56bcc649d39046
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
  Created wheel for unidic-lite: filename=unidic_lite-1.0.8-py3-none-any.whl size=47658817 sha256=60c72e16110a0a6290f59a0cbb1c8704fc6af0368fa226470ee974db9b31bc62
  Stored in directory: /root/.cache/pip/wheels/5e/1f/0f/4d43887e5476d956fae828ee9b6687becd5544d68b51ed633d
Successfully built seqeval unidic-lite


## インポート

In [3]:
import csv
import logging
from collections import defaultdict
from collections.abc import Callable

import numpy as np
import numpy.typing as npt
from datasets import Dataset, DatasetDict
from pydantic.dataclasses import dataclass
from seqeval.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.metrics import classification_report as sklearn_classification_report
from sklearn.metrics import confusion_matrix
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
    EvalPrediction,
    PreTrainedTokenizerBase,
    Trainer,
    TrainerCallback,
    TrainerControl,
    TrainerState,
    TrainingArguments,
)
from transformers.modeling_utils import PreTrainedModel
from transformers.tokenization_utils_base import BatchEncoding

import wandb

In [4]:
from google.colab import userdata

## ロガー設定

In [5]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## BIO ラベル定義

In [6]:
# 7 クラス BIO タグ。NONE はスパンとしてアノテーションせず O タグとして扱う
LABEL_NAMES: tuple[str, ...] = (
    "O",
    "B-PROFILE",
    "I-PROFILE",
    "B-PII",
    "I-PII",
    "B-QPII",
    "I-QPII",
)
LABEL2ID: dict[str, int] = {name: idx for idx, name in enumerate(LABEL_NAMES)}
ID2LABEL: dict[int, str] = {idx: name for idx, name in enumerate(LABEL_NAMES)}

# NONE は O タグとして扱うためマッピングに含めない
ENTITY_TYPES: frozenset[str] = frozenset({"PROFILE", "PII", "QPII"})

## EpochLoggingCallback

In [7]:
class EpochLoggingCallback(TrainerCallback):
    """Epoch の開始・終了をログ出力するコールバック。"""

    def on_epoch_begin(
        self,
        args: TrainingArguments,
        state: TrainerState,
        control: TrainerControl,
        **kwargs,
    ) -> None:
        logger.info(
            "===== Epoch %d/%d 開始 =====",
            int(state.epoch) + 1,
            int(args.num_train_epochs),
        )

    def on_epoch_end(
        self,
        args: TrainingArguments,
        state: TrainerState,
        control: TrainerControl,
        **kwargs,
    ) -> None:
        logger.info(
            "===== Epoch %d/%d 終了 =====",
            int(state.epoch),
            int(args.num_train_epochs),
        )

## Config

In [8]:
@dataclass(frozen=True)
class Config:
    """SFT の設定。

    Attributes:
        exp: 実験番号。wandb の run name と出力先ディレクトリに使用する。
        pretrained_model_dir: 疑似事前学習済みモデルのディレクトリ。
        data_path: 人手アノテーション CSV のパス。
        output_base_dir: モデル出力のベースディレクトリ。
        wandb_project: wandb のプロジェクト名。
        max_length: トークン最大長。
        num_train_epochs: 学習エポック数。
        per_device_train_batch_size: デバイスあたりのバッチサイズ。
        gradient_accumulation_steps: 勾配累積ステップ数。
        learning_rate: 学習率。
        weight_decay: 重み減衰。
        warmup_ratio: 学習率ウォームアップの割合。
        fp16: FP16 混合精度学習を有効にするか。
        dataloader_num_workers: DataLoader のワーカー数。
        eval_ratio: 評価データの割合。
        seed: 乱数シード。
        logging_steps: ログ出力間隔（ステップ数）。
        eval_strategy: 評価戦略。
        save_strategy: チェックポイント保存戦略。
        save_total_limit: 保存するチェックポイントの最大数。
        metric_for_best_model: ベストモデル判定に使うメトリクス名。
        report_to: 学習メトリクスのレポート先。
        early_stopping_patience: Early stopping の patience。
    """

    exp: str = "0009"
    pretrained_model_dir: str = "/content/drive/MyDrive/Colab Notebooks/outputs/0005"
    data_path: str = "/content/drive/MyDrive/Colab Notebooks/sft_annotations.csv"
    output_base_dir: str = "drive/My Drive/Colab Notebooks/outputs"
    wandb_project: str = "sft"
    max_length: int = 128
    num_train_epochs: int = 20
    per_device_train_batch_size: int = 8
    gradient_accumulation_steps: int = 2
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    fp16: bool = True
    dataloader_num_workers: int = 4
    eval_ratio: float = 0.1
    seed: int = 42
    logging_steps: int = 10
    eval_strategy: str = "epoch"
    save_strategy: str = "epoch"
    save_total_limit: int = 1
    metric_for_best_model: str = "macro_f1"
    report_to: str = "wandb"
    early_stopping_patience: int = 5

## データ読み込み関数

In [9]:
def load_sft_annotations(data_path: str) -> Dataset:
    """人手アノテーション CSV を読み込み、テキスト単位でグループ化した Dataset を返す。

    Args:
        data_path: CSV ファイルパス。text, candidate, label カラムを持つ。

    Returns:
        text と spans カラムを持つ Dataset。
    """
    grouped: dict[str, list[dict[str, str]]] = defaultdict(list)
    all_texts: set[str] = set()

    with open(data_path, encoding="utf-8") as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            text: str = row.get("text") or row["sentence"]
            label: str = row["label"]
            candidate: str = row["candidate"]
            all_texts.add(text)

            if label != "NONE":
                grouped[text].append({"candidate": candidate, "label": label})

    records: list[dict] = []
    for text in all_texts:
        records.append({"text": text, "spans": grouped.get(text, [])})

    dataset: Dataset = Dataset.from_list(records)
    logger.info("loaded %d unique texts from %s", len(dataset), data_path)

    label_counts: dict[str, int] = defaultdict(int)
    for record in records:
        if not record["spans"]:
            label_counts["NONE (text-level)"] += 1
        for span in record["spans"]:
            label_counts[span["label"]] += 1
    for label_name, count in sorted(label_counts.items()):
        logger.info("  %s: %d", label_name, count)

    return dataset

## トークンレベルマッチング関数

In [10]:
def find_label_token_spans(
    input_ids: list[int],
    label_ids: list[int],
) -> tuple[tuple[int, int], ...]:
    """入力トークン列からラベルトークン列の出現位置を検索する。

    Args:
        input_ids: テキスト全体のトークン ID 列。
        label_ids: ラベルのトークン ID 列（特殊トークン除去済み）。

    Returns:
        トークンレベルの (start, end) スパンのタプル。
    """
    spans: list[tuple[int, int]] = []
    label_length: int = len(label_ids)
    if label_length == 0:
        return ()
    for start_idx in range(len(input_ids) - label_length + 1):
        if input_ids[start_idx : start_idx + label_length] == label_ids:
            spans.append((start_idx, start_idx + label_length))
    return tuple(spans)

## 型付き BIO タグ割り当て関数

In [11]:
def assign_typed_bio_tags(
    seq_len: int,
    typed_spans: tuple[tuple[int, int, str], ...],
    special_mask: list[bool],
) -> list[int]:
    """トークン列にエンティティ型付き BIO タグを割り当てる。

    Args:
        seq_len: トークン列の長さ。
        typed_spans: (start, end, entity_type) のタプル。
        special_mask: 特殊トークンのマスク（True = 特殊トークン）。

    Returns:
        各トークンに対応する BIO タグ ID のリスト。
    """
    labels: list[int] = [LABEL2ID["O"]] * seq_len
    for start, end, entity_type in typed_spans:
        b_tag: str = f"B-{entity_type}"
        i_tag: str = f"I-{entity_type}"
        labels[start] = LABEL2ID[b_tag]
        for token_idx in range(start + 1, end):
            labels[token_idx] = LABEL2ID[i_tag]
    for token_idx, is_special in enumerate(special_mask):
        if is_special:
            labels[token_idx] = -100
    return labels

## SFT NER データセット構築関数

In [12]:
def create_sft_ner_dataset(
    ds: Dataset,
    tokenizer: PreTrainedTokenizerBase,
    max_length: int,
) -> tuple[Dataset, dict[str, float]]:
    """アノテーション Dataset を NER 学習用 Dataset に変換する。

    Args:
        ds: text と spans カラムを持つ Dataset。
        tokenizer: トークナイザ。
        max_length: トークン最大長。

    Returns:
        NER Dataset と、スパンマッチ統計の辞書のタプル。
    """
    special_ids: set[int] = set(tokenizer.all_special_ids)
    total_labels: int = 0
    matched_labels: int = 0

    def _process(examples: dict[str, list]) -> dict[str, list]:
        nonlocal total_labels, matched_labels

        texts: list[str] = examples["text"]
        spans_list: list[list[dict[str, str]]] = examples["spans"]

        tokenized: BatchEncoding = tokenizer(
            texts,
            max_length=max_length,
            truncation=True,
            padding=False,
        )

        all_labels: list[list[int]] = []
        for example_idx, spans in enumerate(spans_list):
            input_ids: list[int] = tokenized["input_ids"][example_idx]
            special_mask: list[bool] = [tid in special_ids for tid in input_ids]

            all_typed_spans: list[tuple[int, int, str]] = []
            for span_info in spans:
                candidate_text: str = span_info["candidate"]
                entity_type: str = span_info["label"]
                total_labels += 1

                label_encoded: list[int] = tokenizer.encode(
                    candidate_text, add_special_tokens=False
                )
                token_spans: tuple[tuple[int, int], ...] = find_label_token_spans(
                    input_ids, label_encoded
                )
                if token_spans:
                    matched_labels += 1
                    for start, end in token_spans:
                        all_typed_spans.append((start, end, entity_type))

            token_labels: list[int] = assign_typed_bio_tags(
                len(input_ids), tuple(all_typed_spans), special_mask
            )
            all_labels.append(token_labels)

        tokenized["labels"] = all_labels
        return tokenized

    ner_ds: Dataset = ds.map(
        _process,
        batched=True,
        remove_columns=ds.column_names,
        desc="Creating SFT NER dataset",
    )

    match_rate: float = matched_labels / total_labels * 100 if total_labels > 0 else 0.0
    logger.info(
        "span match rate: %d/%d (%.1f%%)",
        matched_labels,
        total_labels,
        match_rate,
    )

    stats: dict[str, float] = {
        "span_match/total_labels": total_labels,
        "span_match/matched_labels": matched_labels,
        "span_match/rate": match_rate,
    }
    return ner_ds, stats

## メトリクス関数

In [13]:
def make_compute_metrics(
    label_names: tuple[str, ...],
) -> Callable[[EvalPrediction], dict[str, float]]:
    """seqeval ベースのメトリクス関数を返す。

    Args:
        label_names: ラベル名のタプル（7 クラス BIO タグ）。

    Returns:
        EvalPrediction を受け取りメトリクス辞書を返す関数。
    """

    def compute_metrics(eval_pred: EvalPrediction) -> dict[str, float]:
        predictions: npt.NDArray[np.intp] = np.argmax(eval_pred.predictions, axis=-1)
        labels: npt.NDArray[np.intp] = eval_pred.label_ids

        true_labels: list[list[str]] = []
        pred_labels: list[list[str]] = []

        for pred_seq, label_seq in zip(predictions, labels):
            true_seq: list[str] = []
            pred_seq_filtered: list[str] = []
            for pred_id, label_id in zip(pred_seq, label_seq):
                if label_id == -100:
                    continue
                true_seq.append(label_names[label_id])
                pred_seq_filtered.append(label_names[pred_id])
            true_labels.append(true_seq)
            pred_labels.append(pred_seq_filtered)

        logger.info("\n%s", classification_report(true_labels, pred_labels))

        report_dict: dict = classification_report(
            true_labels, pred_labels, output_dict=True, zero_division=0
        )

        metrics: dict[str, float] = {
            "precision": precision_score(true_labels, pred_labels),
            "recall": recall_score(true_labels, pred_labels),
            "f1": f1_score(true_labels, pred_labels),
            "macro_f1": f1_score(true_labels, pred_labels, average="macro"),
        }

        for entity_type in ENTITY_TYPES:
            if entity_type in report_dict:
                entity_scores: dict = report_dict[entity_type]
                metrics[f"{entity_type}/precision"] = entity_scores["precision"]
                metrics[f"{entity_type}/recall"] = entity_scores["recall"]
                metrics[f"{entity_type}/f1"] = entity_scores["f1-score"]

        return metrics

    return compute_metrics

## Candidate 単位の 4 クラス分類評価関数

In [14]:
def evaluate_candidate_classification(
    trainer: Trainer,
    eval_dataset: Dataset,
    eval_annotations: list[dict[str, str]],
    eval_dataset_texts: list[str],
    tokenizer: PreTrainedTokenizerBase,
    max_length: int,
) -> None:
    """BIO 予測を candidate 単位の 4 クラスに変換して評価する。

    Args:
        trainer: 学習済み Trainer。
        eval_dataset: トークナイズ済み eval Dataset。
        eval_annotations: eval split のアノテーション。
        eval_dataset_texts: eval_dataset と同じ順序のテキストリスト。
            pred_ids[i] が eval_dataset_texts[i] に対応する。
        tokenizer: トークナイザ。
        max_length: トークン最大長。
    """
    predictions_output = trainer.predict(eval_dataset)
    pred_logits: npt.NDArray = predictions_output.predictions
    pred_ids: npt.NDArray[np.intp] = np.argmax(pred_logits, axis=-1)
    label_ids: npt.NDArray[np.intp] = predictions_output.label_ids

    # Token-level classification report を出力
    true_token_labels: list[list[str]] = []
    pred_token_labels: list[list[str]] = []
    for pred_seq, label_seq in zip(pred_ids, label_ids):
        true_seq: list[str] = []
        pred_seq_filtered: list[str] = []
        for pred_id, label_id in zip(pred_seq, label_seq):
            if label_id == -100:
                continue
            true_seq.append(LABEL_NAMES[label_id])
            pred_seq_filtered.append(LABEL_NAMES[pred_id])
        true_token_labels.append(true_seq)
        pred_token_labels.append(pred_seq_filtered)

    token_report: str = classification_report(true_token_labels, pred_token_labels)
    print("\n=== Token-level Classification Report ===")
    print(token_report)

    # テキスト → pred_ids のインデックスマッピング
    # eval_dataset_texts は eval_dataset と同じ順序なので、
    # pred_ids[i] が eval_dataset_texts[i] のテキストに対応する
    text_to_pred_idx: dict[str, int] = {}
    for idx, text in enumerate(eval_dataset_texts):
        text_to_pred_idx[text] = idx

    true_labels: list[str] = []
    pred_label_list: list[str] = []
    target_names: list[str] = ["NONE", "PII", "PROFILE", "QPII"]

    for annotation in eval_annotations:
        text = annotation["text"]
        candidate: str = annotation["candidate"]
        true_label: str = annotation["label"]

        pred_idx: int | None = text_to_pred_idx.get(text)
        if pred_idx is None:
            continue

        input_ids: list[int] = tokenizer.encode(
            text, max_length=max_length, truncation=True
        )
        label_encoded: list[int] = tokenizer.encode(candidate, add_special_tokens=False)
        token_spans: tuple[tuple[int, int], ...] = find_label_token_spans(
            input_ids, label_encoded
        )

        predicted_label: str = "NONE"
        if token_spans:
            pred_sequence: npt.NDArray = pred_ids[pred_idx]
            entity_votes: dict[str, int] = defaultdict(int)
            for start, end in token_spans:
                for token_pos in range(start, min(end, len(pred_sequence))):
                    tag_id: int = int(pred_sequence[token_pos])
                    tag_name: str = ID2LABEL.get(tag_id, "O")
                    if tag_name.startswith("B-") or tag_name.startswith("I-"):
                        entity: str = tag_name[2:]
                        entity_votes[entity] += 1

            if entity_votes:
                predicted_label = max(entity_votes, key=entity_votes.get)

        true_labels.append(true_label)
        pred_label_list.append(predicted_label)

    report: str = sklearn_classification_report(
        true_labels,
        pred_label_list,
        labels=target_names,
        zero_division=0,
    )
    print("\n=== Candidate-level 4-class Classification Report ===")
    print(report)

    conf_matrix = confusion_matrix(
        true_labels,
        pred_label_list,
        labels=target_names,
    )
    print("=== Confusion Matrix (rows=true, cols=pred) ===")
    print(f"Labels: {target_names}")
    print(conf_matrix)

    conf_data: list[list] = []
    for row_idx, row_label in enumerate(target_names):
        for col_idx, col_label in enumerate(target_names):
            conf_data.append([row_label, col_label, int(conf_matrix[row_idx, col_idx])])

    wandb.log(
        {
            "candidate_classification/report": wandb.Table(
                columns=["metric"],
                data=[[report]],
            ),
            "candidate_classification/confusion_matrix": wandb.Table(
                columns=["true", "predicted", "count"],
                data=conf_data,
            ),
        }
    )

## wandb 認証・設定初期化

In [15]:
config = Config()
output_dir: str = f"{config.output_base_dir}/{config.exp}"

wandb.login(userdata.get('WANDB_API_KEY'))
wandb.login()

wandb.init(
    project=config.wandb_project,
    name=f"exp{config.exp}",
    group=f"exp{config.exp}",
    config={
        "exp": config.exp,
        "pretrained_model_dir": config.pretrained_model_dir,
        "data_path": config.data_path,
        "max_length": config.max_length,
        "num_train_epochs": config.num_train_epochs,
        "per_device_train_batch_size": config.per_device_train_batch_size,
        "gradient_accumulation_steps": config.gradient_accumulation_steps,
        "learning_rate": config.learning_rate,
        "weight_decay": config.weight_decay,
        "warmup_ratio": config.warmup_ratio,
        "fp16": config.fp16,
        "eval_ratio": config.eval_ratio,
        "seed": config.seed,
        "early_stopping_patience": config.early_stopping_patience,
    },
)

logger.info("pretrained_model_dir: %s", config.pretrained_model_dir)
logger.info("data_path: %s", config.data_path)
logger.info("output_dir: %s", output_dir)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nnnnnn-4649-aki (nnnnnn-4649-aki-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## データセット読み込み・NER データセット構築

In [16]:
ds: Dataset = load_sft_annotations(config.data_path)
logger.info("unique texts: %d", len(ds))

tokenizer: PreTrainedTokenizerBase = AutoTokenizer.from_pretrained(
    config.pretrained_model_dir
)

ner_ds: Dataset
span_stats: dict[str, float]
ner_ds, span_stats = create_sft_ner_dataset(ds, tokenizer, config.max_length)
logger.info("NER dataset: %d examples", len(ner_ds))

Parameter 'function'=<function create_sft_ner_dataset.<locals>._process at 0x78d6a00f9a80> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Creating SFT NER dataset:   0%|          | 0/575 [00:00<?, ? examples/s]

## Train / Eval 分割

In [17]:
split: DatasetDict = ner_ds.train_test_split(
    test_size=config.eval_ratio, seed=config.seed
)
train_ds: Dataset = split["train"]
eval_ds: Dataset = split["test"]
logger.info("train: %d, eval: %d", len(train_ds), len(eval_ds))

wandb.log(
    {
        "dataset/total_texts": len(ds),
        "dataset/ner_examples": len(ner_ds),
        "dataset/train_size": len(train_ds),
        "dataset/eval_size": len(eval_ds),
        **span_stats,
    }
)

## モデル・Trainer セットアップ

In [18]:
model: PreTrainedModel = AutoModelForTokenClassification.from_pretrained(
    config.pretrained_model_dir,
    num_labels=len(LABEL_NAMES),
    label2id=LABEL2ID,
    id2label=ID2LABEL,
    ignore_mismatched_sizes=True,
)

training_args: TrainingArguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=config.num_train_epochs,
    per_device_train_batch_size=config.per_device_train_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    weight_decay=config.weight_decay,
    warmup_ratio=config.warmup_ratio,
    fp16=config.fp16,
    dataloader_num_workers=config.dataloader_num_workers,
    eval_strategy=config.eval_strategy,
    save_strategy=config.save_strategy,
    save_total_limit=config.save_total_limit,
    logging_steps=config.logging_steps,
    seed=config.seed,
    metric_for_best_model=config.metric_for_best_model,
    load_best_model_at_end=True,
    report_to=config.report_to,
)

data_collator: DataCollatorForTokenClassification = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)
compute_metrics: Callable[[EvalPrediction], dict[str, float]] = make_compute_metrics(
    LABEL_NAMES
)

early_stopping: EarlyStoppingCallback = EarlyStoppingCallback(
    early_stopping_patience=config.early_stopping_patience,
)

trainer: Trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
    callbacks=[early_stopping, EpochLoggingCallback()],
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: /content/drive/MyDrive/Colab Notebooks/outputs/0005
Key               | Status   |                                                                                     
------------------+----------+-------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([3, 768]) vs model:torch.Size([7, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([3]) vs model:torch.Size([7])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


## 学習実行

In [19]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Macro F1,Pii/precision,Pii/recall,Pii/f1,Profile/precision,Profile/recall,Profile/f1,Qpii/precision,Qpii/recall,Qpii/f1
1,2.328206,1.005775,0.048295,0.101190,0.065385,0.023760,0.000000,0.000000,0.000000,0.048295,0.136000,0.071279,0.000000,0.000000,0.000000
2,1.597139,0.766174,0.262911,0.333333,0.293963,0.300679,0.702703,0.702703,0.702703,0.170455,0.240000,0.199336,0.000000,0.000000,0.000000
3,1.271080,0.644576,0.299674,0.547619,0.387368,0.276518,0.325581,0.756757,0.455285,0.294931,0.512000,0.374269,0.000000,0.000000,0.000000
4,0.934687,0.626606,0.515152,0.607143,0.557377,0.466473,0.894737,0.918919,0.906667,0.450331,0.544000,0.492754,0.000000,0.000000,0.000000
5,0.787037,0.619005,0.504673,0.642857,0.565445,0.467195,0.871795,0.918919,0.894737,0.443114,0.592000,0.506849,0.000000,0.000000,0.000000
6,0.458922,0.760148,0.529412,0.696429,0.601542,0.545397,0.947368,0.972973,0.960000,0.457143,0.640000,0.533333,0.125000,0.166667,0.142857
7,0.382148,0.914960,0.586957,0.642857,0.613636,0.550575,0.947368,0.972973,0.960000,0.510791,0.568000,0.537879,0.142857,0.166667,0.153846
8,0.292516,0.940489,0.618497,0.636905,0.627566,0.600893,0.923077,0.972973,0.947368,0.543307,0.552000,0.547619,0.285714,0.333333,0.307692
9,0.188621,1.086174,0.617647,0.750000,0.677419,0.583205,0.925000,1.000000,0.961039,0.556962,0.704000,0.621908,0.166667,0.166667,0.166667
10,0.176041,1.368008,0.636905,0.636905,0.636905,0.659244,0.948718,1.000000,0.973684,0.549180,0.536000,0.542510,0.428571,0.500000,0.461538


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=561, training_loss=0.5646685992770972, metrics={'train_runtime': 194.134, 'train_samples_per_second': 53.262, 'train_steps_per_second': 3.4, 'total_flos': 463959998294562.0, 'train_loss': 0.5646685992770972, 'epoch': 17.0})

## Candidate 単位の 4 クラス分類評価

In [20]:
eval_indices: list[int] = (
    split["test"]._indices.column(0).to_pylist()
    if split["test"]._indices is not None
    else list(range(len(split["test"])))
)

eval_text_set: set[str] = set()
for idx in eval_indices:
    eval_text_set.add(ds[idx]["text"])

eval_annotations: list[dict[str, str]] = []
with open(config.data_path, encoding="utf-8") as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        row_text: str = row.get("text") or row["sentence"]
        if row_text in eval_text_set:
            eval_annotations.append(
                {
                    "text": row_text,
                    "candidate": row["candidate"],
                    "label": row["label"],
                }
            )

if eval_annotations:
    # eval_dataset と同じ順序のテキストリストを構築する
    # pred_ids[i] が eval_dataset_texts[i] に対応する
    eval_dataset_texts: list[str] = [ds[idx]["text"] for idx in eval_indices]

    evaluate_candidate_classification(
        trainer=trainer,
        eval_dataset=eval_ds,
        eval_annotations=eval_annotations,
        eval_dataset_texts=eval_dataset_texts,
        tokenizer=tokenizer,
        max_length=config.max_length,
    )


=== Token-level Classification Report ===
              precision    recall  f1-score   support

         PII       0.95      1.00      0.97        37
     PROFILE       0.62      0.74      0.67       125
        QPII       0.67      0.67      0.67         6

   micro avg       0.69      0.79      0.73       168
   macro avg       0.74      0.80      0.77       168
weighted avg       0.69      0.79      0.74       168


=== Candidate-level 4-class Classification Report ===
              precision    recall  f1-score   support

        NONE       0.78      0.68      0.72       122
         PII       0.95      1.00      0.97        37
     PROFILE       0.73      0.81      0.77       125
        QPII       0.67      0.67      0.67         6

    accuracy                           0.78       290
   macro avg       0.78      0.79      0.78       290
weighted avg       0.78      0.78      0.77       290

=== Confusion Matrix (rows=true, cols=pred) ===
Labels: ['NONE', 'PII', 'PROFILE', 'QP

## モデル保存・wandb 終了

In [21]:
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
logger.info("model saved to %s", output_dir)

wandb.finish()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

dataset/eval_size,▁
dataset/ner_examples,▁
dataset/total_texts,▁
dataset/train_size,▁
eval/PII/f1,▁▆▄▇▇████████████
eval/PII/precision,▁▆▃▇▇████████████
eval/PII/recall,▁▆▆▇▇████████████
eval/PROFILE/f1,▁▂▅▆▆▆▆▇▇▆▇███▇██
eval/PROFILE/precision,▁▂▄▆▅▆▆▇▇▇▆▇▇█▇▇▇
eval/PROFILE/recall,▁▂▅▆▆▇▆▆█▆▆█▇▇▆▇▇
+36,...
